In [1]:
import gzip
import shutil

with gzip.open("workload.csv.gz", "rb") as f_in:
    with open("workload.csv", "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

In [7]:
import pandas as pd
import numpy as np

# ============================================
# 1. Cargar workload.csv
# ============================================

data_path = 'workload.csv'

headers = [
    'vmid',
    'subscriptionid',
    'deploymentid',
    'vmcreated',
    'vmdeleted',
    'maxcpu',
    'avgcpu',
    'p95maxcpu',
    'vmcategory',
    'vmcorecountbucket',
    'vmmemorybucket'
]

trace_dataframe = pd.read_csv(
    data_path,
    header=None,
    index_col=False,
    names=headers,
    delimiter=','
)


# ============================================
# 2. Convertir timestamps a numérico
# ============================================

trace_dataframe['vmcreated'] = pd.to_numeric(
    trace_dataframe['vmcreated'],
    errors='coerce'
)

trace_dataframe['vmdeleted'] = pd.to_numeric(
    trace_dataframe['vmdeleted'],
    errors='coerce'
)


# ============================================
# 3. Calcular VM Lifetime en horas
# ============================================

trace_dataframe['lifetime'] = (
    np.maximum(
        trace_dataframe['vmdeleted'] - trace_dataframe['vmcreated'],
        300
    ) / 3600
)


# ============================================
# 4. Transformar buckets
# ============================================

trace_dataframe['vmcorecountbucket'] = (
    trace_dataframe['vmcorecountbucket']
    .replace('>24', 30)
)

trace_dataframe['vmmemorybucket'] = (
    trace_dataframe['vmmemorybucket']
    .replace('>64', 70)
)


# ============================================
# 5. Convertir buckets a enteros
# ============================================

trace_dataframe['vmcorecountbucket'] = pd.to_numeric(
    trace_dataframe['vmcorecountbucket'],
    errors='coerce'
).astype('Int64')

trace_dataframe['vmmemorybucket'] = pd.to_numeric(
    trace_dataframe['vmmemorybucket'],
    errors='coerce'
).astype('Int64')


# ============================================
# 6. Calcular Core-Hour
# ============================================

trace_dataframe['corehour'] = (
    trace_dataframe['lifetime']
    * trace_dataframe['vmcorecountbucket']
)


# ============================================
# 7. Resultado
# ============================================

trace_dataframe.head()

,vmid,subscriptionid,deploymentid,vmcreated,vmdeleted,maxcpu,avgcpu,p95maxcpu,vmcategory,vmcorecountbucket,vmmemorybucket,lifetime,corehour
0,71fJw0x+SDRdAxKPwLyHZhTgQpYw2afS6tjJhfT6kHnmLH...,GB6uQC1NSArW5n+TtOybL7GQ1yByjuWtZnsj+5QccZ525R...,2sh/ZjaYdfpslv4iYBfNzFe4rs982kHVvNGJGeQ8MIBCDr...,558300,1673700,91.776885,0.728879,20.759630,Delay-insensitive,8,32,309.833333,2478.666667
1,rKggHO/04j31UFy65mDTwtjdMQL/G03xWfl3xGeiilB4/W...,ub4ty8ygwOECrIz7eaZ/9hDwnCsERvZ3nJJ03sDSpD85et...,+ZraIDUNaWYDZMBiBtZm7xSjr+j3zcHGjup1+wyKxHFmyJ...,424500,425400,37.879261,3.325358,37.879261,Unknown,4,32,0.250000,1.0
2,YrR8gPtBmfNaOdnNEW5If1SdTqQgGQHEnLHGPjySt53bKW...,9LrdYRcUfGbmL2fFfLR/JUg2OTkjGRe3iluwIhDRPnPDPa...,GEyIElfPSFupze8T+T1niQMepeqG88VpLNuxUMyIDbz8VF...,1133100,1133700,0.304368,0.220553,0.304368,Unknown,4,32,0.166667,0.666667
3,xzQ++JF1UAkh70CDhmzkiOo+DQn+E2TLErCFKEmSswv1pl...,0XnZZ8sMN5HY+Yg+0dykYB5oenlgsrCpzpgFSvn/MX42Ze...,7aCQS6fPUw9rwCPiqvghk/WCEbMV3KgNJjA+sssdfY5Ybl...,0,2591400,98.573424,30.340054,98.212503,Interactive,2,4,719.833333,1439.666667
4,vZEivnhabRmImDr+JqKqZnpIM3WxtypwoxjfjnklR/idyR...,HUGaZ+piPP4eHjycCBki2yq0raJywdzrVuriR6nQceH3hA...,/s/D5VtTQDxyS6wq7N/VQAMczx61Ny1Ut3a3iFmDSOCXxp...,228300,229800,82.581449,13.876299,82.581449,Unknown,2,4,0.416667,0.833333


vmcreated + vmdeleted
        =
     lifetime
        ↓
   lifetime × cores
        =
     corehour

In [9]:
trace_dataframe.to_csv("workload_transformed.csv")